# aw_08_b5 — Stage B5: PlayWorld DPO from the B4 policy (Track B, §5.1)

**Parent = B4** (`20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6`,
output sha256:378ba470...). Mirrors A2 exactly (same mining recipe, budget,
DPO profile); the only difference is the parent line (base→B1v2→B4 vs base→A1).
Primary readouts: B5 vs B4 (does DPO still help after two-stage SFT?) and
B5 vs A2 (two-stage vs direct under the same DPO recipe).

**Entry gate: x15 (cell below) must return SEMANTICALLY IDENTICAL.** Otherwise
stop and retrain B4 on the frozen A1-era artifact first.

Cell order: x15 gate → fetch B4 → `a_b5_data` → `b_b5_train` → `c_b5_eval` →
`x09g_run_audit` → `f_b5_analysis`.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title x15 gate — SFT data parity audit (CPU; MUST PASS before any B5 spend)
# Frozen A1-era artifact vs the aw_07 rebuild. Adjust --path if the artifact
# lives elsewhere in m97j/aw_playworld.
!python scripts/fetch_dataset.py \
  --repo m97j/aw_playworld \
  --path preference_train/v1/playworld_sft.jsonl \
  --output data/frozen/playworld_sft_a1era.jsonl

!python scripts/build_training_data.py \
  --seed 1042 --scenarios-per-family 400 --output-dir data/train

!python scripts/x15_sft_data_diff.py \
  --file-a data/frozen/playworld_sft_a1era.jsonl --label-a a1-era-frozen \
  --file-b data/train/playworld_sft.jsonl --label-b rebuild \
  --out runs/x15_sft_data_diff.json


In [ ]:
# @title fetch B4 parent — adapter + lineage sha
B4_RUN_ID = "20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_RUN_ID}
b4_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b4_sha = json.load(open(f"runs/{B4_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b4_dir, b4_sha)


In [ ]:
# @title a_b5_data — mine verifier-guided pairs from the B4 policy (GPU)
# Identical recipe to A2 mining (8 candidates, T=0.8, margin .10, hybrid rank).
# Report the mining manifest (pair count, decision_counts) in the checklist —
# compare pair_yield with A2's to see how much headroom DPO has left.
!python scripts/build_eval_suites.py --episodes-per-suite 300

!python scripts/mine_playworld_pairs.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b4_dir} \
  --prompt-file data/train/playworld_prompts.jsonl \
  --num-candidates 8 --temperature 0.8 --batch-size 100 \
  --selection-method hybrid_verifier_rank --minimum-margin 0.10 \
  --output data/train/playworld_preference_b4.jsonl \
  --hf-sync-repo m97j/aw_playworld --hf-path-in-repo preference_train/b5-v1


In [ ]:
# @title b_b5_train — DPO from the B4 parent (lineage-verified)
!python scripts/run_experiment.py \
  --config configs/experiments/b5_playworld_dpo.yaml \
  --parent-adapter-dir {b4_dir} \
  --override lineage.parent_run_id={B4_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4_sha} \
  --override data.source.local_path=data/train/playworld_preference_b4.jsonl \
  --hf-sync-repo m97j/aw-runs-b5


In [ ]:
# @title c_b5_eval — B5 adapter on the frozen suites (canonical profile)
B5_RUN_ID = ""  # <- from b_b5_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b5 --run-id {B5_RUN_ID}
b5_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b5_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b5


In [ ]:
# @title x09g_run_audit — termination regression check on the B5 eval (CPU)
B5_EVAL = ""  # <- eval run id from c_b5_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b5 --run-id {B5_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B5_EVAL} --out runs/x09_run_audit_b5.json


In [ ]:
# @title f_b5_analysis — B5 vs B4 (DPO gain) and B5 vs A2 (two-stage vs direct)
B4_EVAL = "20260813-000613--eval-playworld--s42--c49f3a"
A2_EVAL = "20260803-011408--eval-playworld--s42--b6f315"

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B5_EVAL} --label-a b5-dpo \
  --run-b runs/{B4_EVAL} --label-b b4-two-stage \
  --output runs/{B5_EVAL}/analysis_b5_vs_b4.json --hf-sync-repo m97j/aw-runs-b5

!python scripts/run_analysis.py \
  --run-a runs/{B5_EVAL} --label-a b5-dpo \
  --run-b runs/{A2_EVAL} --label-b a2-dpo \
  --output runs/{B5_EVAL}/analysis_b5_vs_a2.json --hf-sync-repo m97j/aw-runs-b5


## Stage checklist
- [ ] **x15 verdict = SEMANTICALLY IDENTICAL** (else stop; retrain B4 on frozen artifact)
- [ ] a_b5_data mining manifest recorded (pairs, yield, decision_counts; compare vs A2)
- [ ] b_b5_train lineage verified (parent sha = B4 output 378ba470...)
- [ ] c_b5_eval per-suite pass_rate recorded; x09g truncation/runaway ≈ 0
- [ ] f_b5_analysis: B5 vs B4 and B5 vs A2 deltas + p recorded
- [ ] Feed §6 Phase-2 selection; next: B6 (GRPO, aw_09) + E-RLOO/E-RANDPAIR per plan
